# GVH Diagonal Cubic 0.3.2 — Weak-Field Prediction Foundation

**Partie C — 0.2C2 weak-field predictions**  
**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2  
**Statut :** prediction-foundation / no observational fit  
**Dépendances conceptuelles :** Articles I–II, chaîne 0.2.23 → 0.2.23.5.1, framework 0.3.1.2 Theory–Data Gates

---

## Objectif

Ce notebook construit la **fondation théorique weak-field** nécessaire avant toute confrontation observationnelle.

Il ne doit pas :

- charger des données observationnelles pour fixer la forme d'une correction GVH ;
- transformer une compatibilité avec GR en nouvelle prédiction ;
- promouvoir un paramètre ajusté en quantité `TESTABLE_GVH` ;
- supposer que le secteur timelike est physiquement fermé alors que son origine unique n'est pas encore dérivée.

La question centrale est :

\[
\boxed{
\text{les résultats GVH déjà dérivés suffisent-ils à produire une prédiction weak-field indépendante ?}
}
\]

Le notebook doit pouvoir répondre **NON / BLOCKED** sans modifier la théorie pour forcer un résultat.


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

NOTEBOOK_ID = "GVH_Diagonal_Cubic_0.3.2"
NOTEBOOK_VERSION = "0.3.2"
PART_C_SUBFOLDER = "0.2C2_weak_field_predictions"
AUTHOR = "Charlemagne O Laurince"
EXECUTION_UTC = datetime.now(timezone.utc).isoformat()

ENVIRONMENT_INFO = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "part_c_subfolder": PART_C_SUBFOLDER,
    "author": AUTHOR,
    "execution_utc": EXECUTION_UTC,
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sympy_version": sp.__version__,
}

pd.DataFrame([ENVIRONMENT_INFO]).T.rename(columns={0: "value"})


,value
notebook_id,GVH_Diagonal_Cubic_0.3.2
notebook_version,0.3.2
part_c_subfolder,0.2C2_weak_field_predictions
author,Charlemagne O Laurince
execution_utc,2026-08-08T01:30:51.504044+00:00
python_version,"3.12.13 (main, Mar 4 2026, 09:23:07) [GCC 11...."
platform,Linux-6.6.122+-x86_64-with-glibc2.35
numpy_version,2.0.2
pandas_version,2.2.2
sympy_version,1.14.0


# 1. Règles héritées de 0.3.1.2

Une vraie prédiction weak-field GVH doit satisfaire **THEORY_READY** avant qu'un jeu de données soit utilisé comme test.

Une quantité candidate n'est `TESTABLE_GVH` que si :

1. elle est dérivée d'une structure théorique fixée ;
2. sa forme est indépendante des données utilisées pour la tester ;
3. ses paramètres sont fixés avant le test, ou déterminés par des données indépendantes ;
4. la référence GR est définie ;
5. un modèle d'incertitude est défini ;
6. une règle de falsification est explicitée.

Sinon, le statut doit rester `DERIVED`, `DERIVED_COMPATIBILITY`, `HYPOTHESIS`, `FIT` ou `INCONCLUSIVE`.


In [2]:
ALLOWED_STATUSES = {
    "DERIVED",
    "DERIVED_COMPATIBILITY",
    "REFERENCE_GR",
    "HYPOTHESIS",
    "FIT",
    "TESTABLE_GVH",
    "INCONCLUSIVE",
    "REJECTED",
}

@dataclass
class PredictionCandidate:
    candidate_id: str
    observable: str
    expression: str
    status: str
    theory_source: str
    independent_of_test_data: bool
    parameters_fixed_before_test: bool
    gr_reference_defined: bool
    uncertainty_model_defined: bool
    falsification_rule_defined: bool
    notes: str = ""

    def validate(self):
        if self.status not in ALLOWED_STATUSES:
            raise ValueError(f"Invalid status: {self.status}")
        return True


# 2. Famille métrique weak-field héritée

La chaîne 0.2.22–0.2.23.1 utilise la famille statique isotrope :

\[
g_{tt}^{\rm GVH}
=
-1+2a_1u-2a_2u^2+\mathcal O(u^3),
\]

\[
g_{ij}^{\rm GVH}
=
\left(
1+2b_1u+b_2u^2+\mathcal O(u^3)
\right)\delta_{ij},
\]

avec

\[
u=\frac{G_N M}{r c^2}.
\]

Le matching PPN déjà dérivé pour cette famille paramétrique est :

\[
\boxed{\gamma_{\rm GVH}=\frac{b_1}{a_1}},
\qquad
\boxed{\beta_{\rm GVH}=\frac{a_2}{a_1^2}}.
\]

Ces relations sont des **relations de matching**. Elles ne fixent pas à elles seules les valeurs de \(a_1,a_2,b_1\).


In [3]:
a1, a2, b1, b2, u = sp.symbols("a1 a2 b1 b2 u", real=True)

gtt = -1 + 2*a1*u - 2*a2*u**2
gspace = 1 + 2*b1*u + b2*u**2

gamma_GVH = sp.simplify(b1 / a1)
beta_GVH = sp.simplify(a2 / a1**2)

parametric_matching_df = pd.DataFrame([
    {
        "quantity": "gamma_GVH",
        "expression": str(gamma_GVH),
        "status": "DERIVED_COMPATIBILITY",
        "scope": "parametric static isotropic weak-field family",
    },
    {
        "quantity": "beta_GVH",
        "expression": str(beta_GVH),
        "status": "DERIVED_COMPATIBILITY",
        "scope": "parametric static isotropic weak-field family",
    },
])

parametric_matching_df


,quantity,expression,status,scope
0,gamma_GVH,b1/a1,DERIVED_COMPATIBILITY,parametric static isotropic weak-field family
1,beta_GVH,a2/a1**2,DERIVED_COMPATIBILITY,parametric static isotropic weak-field family


# 3. Déviation par rapport à GR

Définissons seulement des **diagnostics symboliques** :

\[
\Delta\gamma
=
\gamma_{\rm GVH}-1,
\qquad
\Delta\beta
=
\beta_{\rm GVH}-1.
\]

Ces quantités deviennent des prédictions seulement si \(a_1,a_2,b_1\) sont eux-mêmes déterminés par une dynamique GVH fermée.


In [4]:
delta_gamma = sp.simplify(gamma_GVH - 1)
delta_beta = sp.simplify(beta_GVH - 1)

deviation_df = pd.DataFrame([
    {
        "diagnostic": "Delta_gamma",
        "expression": str(delta_gamma),
        "prediction_status": "NOT_YET_A_PREDICTION",
    },
    {
        "diagnostic": "Delta_beta",
        "expression": str(delta_beta),
        "prediction_status": "NOT_YET_A_PREDICTION",
    },
])

deviation_df


,diagnostic,expression,prediction_status
0,Delta_gamma,(-a1 + b1)/a1,NOT_YET_A_PREDICTION
1,Delta_beta,-1 + a2/a1**2,NOT_YET_A_PREDICTION


# 4. Développement autour de la branche GR

Pour étudier la structure algébrique sans fixer arbitrairement la physique :

\[
a_1=1+\epsilon_D\alpha_1,
\qquad
a_2=1+\epsilon_D\alpha_2,
\qquad
b_1=1+\epsilon_D\sigma_1.
\]

À premier ordre :

\[
\Delta\gamma
=
\epsilon_D(\sigma_1-\alpha_1)
+\mathcal O(\epsilon_D^2),
\]

\[
\Delta\beta
=
\epsilon_D(\alpha_2-2\alpha_1)
+\mathcal O(\epsilon_D^2).
\]

Ces expressions indiquent **quelles combinaisons** devraient être déterminées par la théorie ; elles ne fournissent pas encore leurs valeurs.


In [5]:
epsilon_D = sp.symbols("epsilon_D", real=True)
alpha1, alpha2, sigma1 = sp.symbols("alpha1 alpha2 sigma1", real=True)

subs_linear = {
    a1: 1 + epsilon_D*alpha1,
    a2: 1 + epsilon_D*alpha2,
    b1: 1 + epsilon_D*sigma1,
}

delta_gamma_linear = sp.expand(
    sp.series(delta_gamma.subs(subs_linear), epsilon_D, 0, 2).removeO()
)
delta_beta_linear = sp.expand(
    sp.series(delta_beta.subs(subs_linear), epsilon_D, 0, 2).removeO()
)

linearized_deviation_df = pd.DataFrame([
    {"quantity": "Delta_gamma", "expression": str(delta_gamma_linear)},
    {"quantity": "Delta_beta", "expression": str(delta_beta_linear)},
])

linearized_deviation_df


,quantity,expression
0,Delta_gamma,-alpha1*epsilon_D + epsilon_D*sigma1
1,Delta_beta,-2*alpha1*epsilon_D + alpha2*epsilon_D


# 5. Branche restreinte déjà dérivée

Dans la branche précise étudiée dans 0.2.23.1 :

- source parfaite isotrope ;
- couplage strict à la partie spatiale sans trace ;
- \(\Pi_{\mu\nu}[T]=0\) ;
- \(Q_D=0\) ;
- champ directionnel extérieur nul.

Alors l'extérieur est Schwarzschild et :

\[
a_1=a_2=b_1=1.
\]

Donc :

\[
\gamma_{\rm GVH}=1,
\qquad
\beta_{\rm GVH}=1.
\]

**Ce résultat est une récupération conditionnelle de GR, pas une prédiction globale de tout GVH.**


In [6]:
strict_branch = {
    a1: sp.Integer(1),
    a2: sp.Integer(1),
    b1: sp.Integer(1),
}

strict_gamma = sp.simplify(gamma_GVH.subs(strict_branch))
strict_beta = sp.simplify(beta_GVH.subs(strict_branch))

strict_branch_df = pd.DataFrame([{
    "branch": "strict_isotropic_traceless_source",
    "gamma_GVH": str(strict_gamma),
    "beta_GVH": str(strict_beta),
    "Delta_gamma": str(sp.simplify(strict_gamma - 1)),
    "Delta_beta": str(sp.simplify(strict_beta - 1)),
    "status": "DERIVED_COMPATIBILITY",
    "global_GVH_prediction": False,
}])

strict_branch_df


,branch,gamma_GVH,beta_GVH,Delta_gamma,Delta_beta,status,global_GVH_prediction
0,strict_isotropic_traceless_source,1,1,0,0,DERIVED_COMPATIBILITY,False


# 6. Branche directionnelle non triviale

La chaîne théorique antérieure autorise conceptuellement une branche où une correction directionnelle extérieure pourrait être non nulle, par exemple sous une forme de travail :

\[
d(r)=\frac{Q_D}{r}e^{-\mu_D r}.
\]

Une métrique weak-field de prototype peut alors contenir des termes du type

\[
g_{tt}
=
g_{tt}^{\rm GR}+2\alpha_t d(r),
\]

\[
g_{\rm space}
=
g_{\rm space}^{\rm GR}+2\alpha_s d(r).
\]

Mais les Articles I–II et la chaîne 0.2.23 montrent que l'origine physique unique, les couplages \(\alpha_t,\alpha_s\), \(Q_D\), \(\mu_D\), et le secteur non linéaire ne sont pas encore tous dérivés d'une dynamique fondamentale unique.

La forme ci-dessus reste donc un **prototype/hypothèse de travail**, pas une prédiction.


In [7]:
r, Q_D, mu_D, alpha_t, alpha_s = sp.symbols(
    "r Q_D mu_D alpha_t alpha_s",
    positive=True,
    finite=True,
)

d_yukawa = sp.simplify(Q_D * sp.exp(-mu_D*r) / r)
delta_gtt_candidate = sp.simplify(2*alpha_t*d_yukawa)
delta_gspace_candidate = sp.simplify(2*alpha_s*d_yukawa)

nontrivial_prototype_df = pd.DataFrame([
    {
        "quantity": "d(r)",
        "expression": str(d_yukawa),
        "status": "HYPOTHESIS/PROTOTYPE",
    },
    {
        "quantity": "delta_gtt",
        "expression": str(delta_gtt_candidate),
        "status": "HYPOTHESIS/PROTOTYPE",
    },
    {
        "quantity": "delta_gspace",
        "expression": str(delta_gspace_candidate),
        "status": "HYPOTHESIS/PROTOTYPE",
    },
])

nontrivial_prototype_df


,quantity,expression,status
0,d(r),Q_D*exp(-mu_D*r)/r,HYPOTHESIS/PROTOTYPE
1,delta_gtt,2*Q_D*alpha_t*exp(-mu_D*r)/r,HYPOTHESIS/PROTOTYPE
2,delta_gspace,2*Q_D*alpha_s*exp(-mu_D*r)/r,HYPOTHESIS/PROTOTYPE


# 7. Conditions minimales pour transformer la branche non triviale en prédiction

Une correction weak-field non-GR ne peut être promue en `TESTABLE_GVH` que si les éléments suivants sont dérivés ou fixés indépendamment du jeu de données de test.


In [8]:
prediction_requirements_df = pd.DataFrame([
    {
        "requirement": "unique_covariant_dynamics",
        "description": "Une dynamique/action fixée sélectionne la branche physique",
        "status": "MISSING",
    },
    {
        "requirement": "timelike_field_origin",
        "description": "Origine et rôle physique unique de u^mu / secteur timelike",
        "status": "MISSING",
    },
    {
        "requirement": "source_projector_physical_selection",
        "description": "Le projecteur/canal source pertinent doit être sélectionné physiquement",
        "status": "PARTIAL",
    },
    {
        "requirement": "alpha_t",
        "description": "Backreaction temporelle dérivée",
        "status": "MISSING",
    },
    {
        "requirement": "alpha_s",
        "description": "Backreaction spatiale dérivée",
        "status": "MISSING",
    },
    {
        "requirement": "Q_D",
        "description": "Charge directionnelle déterminée par la source/dynamique",
        "status": "MISSING",
    },
    {
        "requirement": "mu_D",
        "description": "Échelle de portée dérivée/fixée indépendamment",
        "status": "MISSING",
    },
    {
        "requirement": "second_order_sector",
        "description": "Termes nécessaires à beta_GVH et au second ordre",
        "status": "MISSING",
    },
    {
        "requirement": "isotropic_gauge_resolution",
        "description": "Séparation explicite entre effet physique et choix de coordonnées",
        "status": "REQUIRED",
    },
    {
        "requirement": "falsification_rule",
        "description": "Règle quantitative définie avant l'analyse des données de test",
        "status": "MISSING",
    },
])

prediction_requirements_df


,requirement,description,status
0,unique_covariant_dynamics,Une dynamique/action fixée sélectionne la bran...,MISSING
1,timelike_field_origin,Origine et rôle physique unique de u^mu / sect...,MISSING
2,source_projector_physical_selection,Le projecteur/canal source pertinent doit être...,PARTIAL
3,alpha_t,Backreaction temporelle dérivée,MISSING
4,alpha_s,Backreaction spatiale dérivée,MISSING
5,Q_D,Charge directionnelle déterminée par la source...,MISSING
6,mu_D,Échelle de portée dérivée/fixée indépendamment,MISSING
7,second_order_sector,Termes nécessaires à beta_GVH et au second ordre,MISSING
8,isotropic_gauge_resolution,Séparation explicite entre effet physique et c...,REQUIRED
9,falsification_rule,Règle quantitative définie avant l'analyse des...,MISSING


# 8. Catalogue des observables weak-field candidates

Le but ici est de définir **où** une correction théorique pourrait apparaître, sans lui attribuer une amplitude non dérivée.

Les observables candidates pour C2 → C4 incluent notamment :

- paramètres PPN \(\gamma,\beta\) ;
- déviation de la lumière ;
- délai de Shapiro ;
- précession du périhélie ;
- redshift gravitationnel ;
- éventuels signaux de référentiel préféré, si un tel secteur est réellement dérivé.

À ce stade, elles sont des **interfaces de test futures**, pas des succès ou échecs observationnels.


In [9]:
observable_catalog_df = pd.DataFrame([
    {
        "observable_id": "PPN_GAMMA",
        "reference_theory": "GR",
        "reference_value": "1",
        "GVH_interface": "b1/a1",
        "current_status": "DERIVED_COMPATIBILITY",
        "testable_now": False,
    },
    {
        "observable_id": "PPN_BETA",
        "reference_theory": "GR",
        "reference_value": "1",
        "GVH_interface": "a2/a1^2",
        "current_status": "DERIVED_COMPATIBILITY",
        "testable_now": False,
    },
    {
        "observable_id": "LIGHT_DEFLECTION",
        "reference_theory": "GR",
        "reference_value": "REFERENCE_FORMULA_TO_BE_IMPORTED_IN_C3",
        "GVH_interface": "depends on gamma_GVH / full weak metric",
        "current_status": "INTERFACE_ONLY",
        "testable_now": False,
    },
    {
        "observable_id": "SHAPIRO_DELAY",
        "reference_theory": "GR",
        "reference_value": "REFERENCE_FORMULA_TO_BE_IMPORTED_IN_C3",
        "GVH_interface": "depends on gamma_GVH / full weak metric",
        "current_status": "INTERFACE_ONLY",
        "testable_now": False,
    },
    {
        "observable_id": "PERIHELION_PRECESSION",
        "reference_theory": "GR",
        "reference_value": "REFERENCE_FORMULA_TO_BE_IMPORTED_IN_C3",
        "GVH_interface": "depends on beta_GVH and gamma_GVH",
        "current_status": "INTERFACE_ONLY",
        "testable_now": False,
    },
    {
        "observable_id": "GRAVITATIONAL_REDSHIFT",
        "reference_theory": "GR",
        "reference_value": "REFERENCE_FORMULA_TO_BE_IMPORTED_IN_C3",
        "GVH_interface": "depends on temporal weak metric",
        "current_status": "INTERFACE_ONLY",
        "testable_now": False,
    },
])

observable_catalog_df


,observable_id,reference_theory,reference_value,GVH_interface,current_status,testable_now
0,PPN_GAMMA,GR,1,b1/a1,DERIVED_COMPATIBILITY,False
1,PPN_BETA,GR,1,a2/a1^2,DERIVED_COMPATIBILITY,False
2,LIGHT_DEFLECTION,GR,REFERENCE_FORMULA_TO_BE_IMPORTED_IN_C3,depends on gamma_GVH / full weak metric,INTERFACE_ONLY,False
3,SHAPIRO_DELAY,GR,REFERENCE_FORMULA_TO_BE_IMPORTED_IN_C3,depends on gamma_GVH / full weak metric,INTERFACE_ONLY,False
4,PERIHELION_PRECESSION,GR,REFERENCE_FORMULA_TO_BE_IMPORTED_IN_C3,depends on beta_GVH and gamma_GVH,INTERFACE_ONLY,False
5,GRAVITATIONAL_REDSHIFT,GR,REFERENCE_FORMULA_TO_BE_IMPORTED_IN_C3,depends on temporal weak metric,INTERFACE_ONLY,False


# 9. Garde-fou contre l'ajustement ad hoc

Une valeur choisie après consultation des données de test ne constitue pas une prédiction.

Cette cellule impose une différence entre :

- paramètre dérivé par la théorie ;
- paramètre fixé par une source indépendante ;
- paramètre ajusté sur les données de test.


In [10]:
def classify_parameter_origin(origin: str) -> str:
    allowed = {
        "THEORY_DERIVED": "ELIGIBLE",
        "INDEPENDENT_CALIBRATION": "CONDITIONALLY_ELIGIBLE",
        "TEST_DATA_FIT": "NOT_A_PREDICTION",
        "UNFIXED": "BLOCKED",
    }
    if origin not in allowed:
        raise ValueError(f"Unknown origin: {origin}")
    return allowed[origin]

parameter_origin_examples_df = pd.DataFrame([
    {"origin": key, "eligibility": classify_parameter_origin(key)}
    for key in ["THEORY_DERIVED", "INDEPENDENT_CALIBRATION", "TEST_DATA_FIT", "UNFIXED"]
])

parameter_origin_examples_df


,origin,eligibility
0,THEORY_DERIVED,ELIGIBLE
1,INDEPENDENT_CALIBRATION,CONDITIONALLY_ELIGIBLE
2,TEST_DATA_FIT,NOT_A_PREDICTION
3,UNFIXED,BLOCKED


# 10. Registre des candidats weak-field

Aucun candidat ne doit être marqué `TESTABLE_GVH` si les coefficients physiques non triviaux restent non dérivés.


In [11]:
CANDIDATES = [
    PredictionCandidate(
        candidate_id="STRICT_GR_BRANCH_PPN",
        observable="gamma_GVH, beta_GVH",
        expression="gamma=1; beta=1 under strict isotropic traceless-source branch",
        status="DERIVED_COMPATIBILITY",
        theory_source="0.2.23.1",
        independent_of_test_data=True,
        parameters_fixed_before_test=True,
        gr_reference_defined=True,
        uncertainty_model_defined=False,
        falsification_rule_defined=False,
        notes="Restricted GR-recovery branch; not a global non-GR GVH prediction.",
    ),
    PredictionCandidate(
        candidate_id="GLOBAL_PPN_DEVIATION",
        observable="Delta_gamma, Delta_beta",
        expression="Delta_gamma=b1/a1-1; Delta_beta=a2/a1^2-1",
        status="INCONCLUSIVE",
        theory_source="0.2.22 + 0.2.23.1 + Article II",
        independent_of_test_data=True,
        parameters_fixed_before_test=False,
        gr_reference_defined=True,
        uncertainty_model_defined=False,
        falsification_rule_defined=False,
        notes="Coefficient mapping exists; unique nontrivial coefficients do not.",
    ),
    PredictionCandidate(
        candidate_id="YUKAWA_DIRECTIONAL_CORRECTION",
        observable="delta_gtt(r), delta_gspace(r)",
        expression="2 alpha_{t,s} Q_D exp(-mu_D r)/r",
        status="HYPOTHESIS",
        theory_source="reduced prototype in 0.2.23 chain",
        independent_of_test_data=True,
        parameters_fixed_before_test=False,
        gr_reference_defined=True,
        uncertainty_model_defined=False,
        falsification_rule_defined=False,
        notes="Prototype only; origin/couplings/range not uniquely derived.",
    ),
]

for candidate in CANDIDATES:
    candidate.validate()

candidate_registry_df = pd.DataFrame([asdict(c) for c in CANDIDATES])
candidate_registry_df


,candidate_id,observable,expression,status,theory_source,independent_of_test_data,parameters_fixed_before_test,gr_reference_defined,uncertainty_model_defined,falsification_rule_defined,notes
0,STRICT_GR_BRANCH_PPN,"gamma_GVH, beta_GVH",gamma=1; beta=1 under strict isotropic tracele...,DERIVED_COMPATIBILITY,0.2.23.1,True,True,True,False,False,Restricted GR-recovery branch; not a global no...
1,GLOBAL_PPN_DEVIATION,"Delta_gamma, Delta_beta",Delta_gamma=b1/a1-1; Delta_beta=a2/a1^2-1,INCONCLUSIVE,0.2.22 + 0.2.23.1 + Article II,True,False,True,False,False,Coefficient mapping exists; unique nontrivial ...
2,YUKAWA_DIRECTIONAL_CORRECTION,"delta_gtt(r), delta_gspace(r)","2 alpha_{t,s} Q_D exp(-mu_D r)/r",HYPOTHESIS,reduced prototype in 0.2.23 chain,True,False,True,False,False,Prototype only; origin/couplings/range not uni...


# 11. Porte THEORY_READY weak-field

La porte n'est ouverte que pour un candidat explicitement `TESTABLE_GVH` avec toutes les dépendances scientifiques satisfaites.


In [12]:
def theory_ready(candidate: PredictionCandidate) -> dict:
    checks = {
        "status_is_TESTABLE_GVH": candidate.status == "TESTABLE_GVH",
        "independent_of_test_data": bool(candidate.independent_of_test_data),
        "parameters_fixed_before_test": bool(candidate.parameters_fixed_before_test),
        "gr_reference_defined": bool(candidate.gr_reference_defined),
        "uncertainty_model_defined": bool(candidate.uncertainty_model_defined),
        "falsification_rule_defined": bool(candidate.falsification_rule_defined),
    }
    checks["THEORY_READY"] = all(checks.values())
    return checks

theory_gate_df = pd.DataFrame([
    {
        "candidate_id": c.candidate_id,
        "status": c.status,
        **theory_ready(c),
    }
    for c in CANDIDATES
])

theory_gate_df


,candidate_id,status,status_is_TESTABLE_GVH,independent_of_test_data,parameters_fixed_before_test,gr_reference_defined,uncertainty_model_defined,falsification_rule_defined,THEORY_READY
0,STRICT_GR_BRANCH_PPN,DERIVED_COMPATIBILITY,False,True,True,True,False,False,False
1,GLOBAL_PPN_DEVIATION,INCONCLUSIVE,False,True,False,True,False,False,False
2,YUKAWA_DIRECTIONAL_CORRECTION,HYPOTHESIS,False,True,False,True,False,False,False


# 12. Verdict de 0.3.2

Trois issues sont permises :

### A. `PASS-RESTRICTED-GR-RECOVERY_ONLY`

La branche restreinte récupère GR, mais aucune correction globale indépendante n'est dérivée.

### B. `PASS-WEAK-FIELD-THEORY-READY`

Au moins une prédiction non-GR indépendante satisfait toutes les conditions de `THEORY_READY`.

### C. `FAIL-INTERNAL-CONSISTENCY`

Une incohérence mathématique ou logique est détectée.

Le verdict actuel attendu est **A**, sauf si de nouvelles dérivations théoriques sont explicitement introduites dans une version ultérieure.


In [13]:
internal_checks = {
    "parametric_matching_symbolic": bool(
        sp.simplify(gamma_GVH - b1/a1) == 0
        and sp.simplify(beta_GVH - a2/a1**2) == 0
    ),
    "strict_gamma_is_GR": bool(strict_gamma == 1),
    "strict_beta_is_GR": bool(strict_beta == 1),
    "strict_branch_not_global_prediction": True,
    "no_candidate_false_promoted_to_TESTABLE_GVH": not any(
        c.status == "TESTABLE_GVH" for c in CANDIDATES
    ),
    "no_theory_gate_open": not bool(theory_gate_df["THEORY_READY"].any()),
}

INTERNAL_PASS = all(internal_checks.values())

if not INTERNAL_PASS:
    FINAL_STATUS = "FAIL-INTERNAL-CONSISTENCY"
elif theory_gate_df["THEORY_READY"].any():
    FINAL_STATUS = "PASS-WEAK-FIELD-THEORY-READY"
else:
    FINAL_STATUS = "PASS-RESTRICTED-GR-RECOVERY_ONLY_BLOCKED-NONTRIVIAL-PREDICTION"

validation_df = pd.DataFrame([
    {"test": key, "pass": value}
    for key, value in internal_checks.items()
])

print("FINAL STATUS:", FINAL_STATUS)
validation_df


FINAL STATUS: PASS-RESTRICTED-GR-RECOVERY_ONLY_BLOCKED-NONTRIVIAL-PREDICTION


,test,pass
0,parametric_matching_symbolic,True
1,strict_gamma_is_GR,True
2,strict_beta_is_GR,True
3,strict_branch_not_global_prediction,True
4,no_candidate_false_promoted_to_TESTABLE_GVH,True
5,no_theory_gate_open,True


# 13. Artefacts exportables

Le notebook exporte des objets de gouvernance weak-field.

Aucun fichier ne doit être nommé comme une prédiction globale canonique tant que `THEORY_READY=False`.


In [14]:
PROJECT_ROOT_CANDIDATES = [
    Path("/content/Univers/gvh_diagonal_cubic"),
    Path.cwd(),
]

PROJECT_ROOT = next(
    (p for p in PROJECT_ROOT_CANDIDATES if p.exists()),
    Path.cwd(),
)

EXPORT_DIR = PROJECT_ROOT / "exports"
PROCESSED_PPN_DIR = PROJECT_ROOT / "data" / "processed" / "ppn"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_PPN_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = "GVH_Diagonal_Cubic_0.3.2"

exports = {
    "Parametric_Matching": parametric_matching_df,
    "Weak_Field_Deviations": deviation_df,
    "Linearized_Deviations": linearized_deviation_df,
    "Strict_GR_Branch": strict_branch_df,
    "Nontrivial_Prototype": nontrivial_prototype_df,
    "Prediction_Requirements": prediction_requirements_df,
    "Observable_Catalog": observable_catalog_df,
    "Candidate_Registry": candidate_registry_df,
    "Theory_Gate": theory_gate_df,
    "Validation": validation_df,
}

for suffix, table in exports.items():
    table.to_csv(EXPORT_DIR / f"{PREFIX}_{suffix}.csv", index=False)

blocked_artifact = {
    "artifact_status": "BLOCKED_NONTRIVIAL_WEAK_FIELD_PREDICTION_NOT_DERIVED",
    "notebook": NOTEBOOK_ID,
    "version": NOTEBOOK_VERSION,
    "final_status": FINAL_STATUS,
    "global_model_prediction": False,
    "derived_interfaces": {
        "gamma_GVH": "b1/a1",
        "beta_GVH": "a2/a1^2",
        "Delta_gamma": "b1/a1 - 1",
        "Delta_beta": "a2/a1^2 - 1",
    },
    "restricted_branch": {
        "conditions": "perfect isotropic source + strict traceless source coupling",
        "gamma_GVH": "1",
        "beta_GVH": "1",
        "interpretation": "restricted GR recovery only",
    },
    "nontrivial_branch": {
        "status": "NOT_THEORY_READY",
        "missing": prediction_requirements_df.loc[
            prediction_requirements_df["status"].isin(["MISSING", "REQUIRED"]),
            "requirement"
        ].tolist(),
    },
    "warning": (
        "Do not use this artifact as a global non-GR GVH prediction. "
        "No observational fitting has been performed in 0.3.2."
    ),
}

BLOCKED_ARTIFACT_FILE = (
    PROCESSED_PPN_DIR
    / "gvh_0.3.2_weak_field_prediction_foundation_BLOCKED.json"
)

BLOCKED_ARTIFACT_FILE.write_text(
    json.dumps(blocked_artifact, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Exports:", EXPORT_DIR)
print("Blocked artifact:", BLOCKED_ARTIFACT_FILE)


Exports: /content/exports
Blocked artifact: /content/data/processed/ppn/gvh_0.3.2_weak_field_prediction_foundation_BLOCKED.json


# 14. Prochaine décision scientifique

## Si `THEORY_READY=False`

Ne pas aller directement chercher une anomalie dans les données pour fixer la théorie.

La prochaine tâche théorique doit résoudre au moins l'un des verrous suivants :

1. sélectionner une dynamique covariante unique ;
2. dériver l'origine du secteur timelike ;
3. dériver les backreactions métriques \(lpha_t,lpha_s\) ;
4. déterminer \(Q_D\) et \(\mu_D\) depuis la théorie/source ;
5. fermer le second ordre nécessaire à \(eta\) ;
6. définir une correction observable pré-enregistrable.

## Si une future version obtient `THEORY_READY=True`

Alors seulement, la chaîne peut continuer vers :

```text
0.2C3_solar_system_tests/
0.2C4_PPN_constraints/
```

avec données réelles et critères de falsification pré-définis.


# Conclusion

Le résultat utile de 0.3.2 n'est pas forcément une nouvelle valeur numérique.

Le notebook établit ce qui est déjà dérivé :

\[
\gamma_{\rm GVH}=\frac{b_1}{a_1},
\qquad
\beta_{\rm GVH}=\frac{a_2}{a_1^2},
\]

et confirme que la branche isotrope restreinte récupère :

\[
\gamma=\beta=1.
\]

Mais il refuse de transformer la branche non triviale en prédiction tant que ses coefficients physiques ne sont pas dérivés indépendamment des données.

Le statut scientifiquement sain attendu est donc :

```text
PASS-RESTRICTED-GR-RECOVERY_ONLY_BLOCKED-NONTRIVIAL-PREDICTION
```

Ce statut signifie :

- la fondation weak-field est cohérente ;
- la récupération GR restreinte est préservée ;
- aucune nouvelle prédiction non-GR n'est encore artificiellement déclarée ;
- le verrou `THEORY_READY` reste explicitement visible.
